# Create a Folium Map of the Results

The results of the main notebook have been saved in `output/labels_geojson/`.
This notebook reads that GeoJSON, smooths the shapes, and renders a Folium web map.

**Prerequisites:** Run `main_notebook.ipynb` first so that these files exist:
- `output/labels_geojson/combined_map_labels.geojson`
- `output/tabular_and_text/representative_positions.pkl`

In [ ]:
import geopandas as gpd
import folium
from folium.plugins import MousePosition
import numpy as np
from shapely.geometry import Polygon, MultiPolygon
from common import replace_umlaute

def round_coords(geom, decimals=4):
    if geom.geom_type == 'Polygon':
        exterior = np.round(np.array(geom.exterior.coords), decimals).tolist()
        interiors = [
            np.round(np.array(ring.coords), decimals).tolist()
            for ring in geom.interiors
        ]
        return Polygon(exterior, interiors)
    
    elif geom.geom_type == 'MultiPolygon':
        parts = []
        for p in geom.geoms:
            exterior = np.round(np.array(p.exterior.coords), decimals).tolist()
            interiors = [
                np.round(np.array(ring.coords), decimals).tolist()
                for ring in p.interiors
            ]
            parts.append(Polygon(exterior, interiors))
        return MultiPolygon(parts)
    
    return geom

## Representative Positions
So we have some representative Positions. This is just a list object containing lat/lon coordinates. 
They have been saved by the main script and are already illustrated in the documentation.

As they change together with different settings I fixated them in this array. They can however be loaded from whatever the main script returns.

In [ ]:
import pickle

with open("output/tabular_and_text/representative_positions.pkl", "rb") as f:
    representative_positions = pickle.load(f)


representative_positions = {
    0: [(12.7796, 45.4373), (11.2246, 54.6149), (0.3724, 52.8735)],
    1: [(11.5235, 54.5613), (-0.0386, 52.5027), (-1.2785, 46.2522)],
    2: [(-1.0541, 53.8857), (11.9864, 52.7643), (13.5773, 53.8230)],
    3: [(3.1497, 50.3580), (-2.4971, 53.1608), (-0.4576, 53.2139)],
    4: [(10.7206, 45.2668), (-0.8134, 52.2503), (4.0064, 43.6073)],
    5: [(8.4562, 44.9047), (-7.2350, 53.9242), (-8.5240, 51.9537)],
    6: [(-2.2159, 42.4004), (-0.5915, 48.8978), (-9.0046, 52.2587)],
    7: [(-3.0215, 42.6649), (-8.9244, 52.0231), (8.5176, 44.5870)],
    8: [(11.5893, 44.0062), (8.9220, 44.5162), (-3.8236, 52.6587)],
    9: [(0.9308, 42.6556), (10.1733, 44.0430), (10.2515, 46.3518)],
}

## Custom colors
I try to keep the colors consistent across the whole project. 
For that reason here’s the color array – I will put it in a sidecar py. #todo

In [ ]:
from geo_colors import custom_colors_dark, custom_colors_medium

## Visual elements of the map
Here I define markers etc.

In [ ]:
# These are color coded label presets marking representative positions on the clusters
x_icon = [
    folium.DivIcon(
        html=f"""
        <div style="
            display: flex; 
            align-items: center; 
            justify-content: center;
            font-size: 60px; 
            color: {single_color}; 
            font-weight: bold;
        ">✕</div>""",
        icon_anchor=(30, 30),
        icon_size=(60, 60),
    )
    for single_color in custom_colors_dark
]

# Marks only two places with a bigger X
x_icon_big = folium.DivIcon(
    html="""
        <div style="
            display: flex; 
            align-items: center; 
            justify-content: center;
            font-size: 100px; 
            color: red; 
            font-weight: bold;
        ">✕</div>""",
    icon_anchor=(100, 100),
    icon_size=(200, 200),
)

# This is a safer way to get colored shapes
# We first convert the list above to a dict
# so we can make use of gray if the index is not found
def style(feature):
    colordict = {index: value for index, value in enumerate(custom_colors_medium)}
    farbe = colordict.get(feature["properties"]["label"], "gray")
    return {"fillColor": farbe, "color": farbe, "fillOpacity": 0.333}


# This here is a trick to put a colored box inside the legend
# That way we can easily find and discern the differenc clusters
def generate_coloured_name(name, colorstring):
    return f'<span><svg width="12" height="12"><rect width="12" height="12" style="fill:{colorstring}"/></svg> {name}</span>'

## Simplify and correct the shapes
The shapes have been created by vectorizing images. Thus they might be a bit pixelated.
First we simplify the shapes by simplifying them. Afterwards we round the coordinates which evidently makes the map html file smaller..

In [ ]:
gdf = gpd.read_file("output/labels_geojson/combined_map_labels.geojson")
tolerance = 0.018

gdf = gdf[gdf.geometry.notna()]

gdf["geometry"] = gdf.simplify_coverage(tolerance=tolerance)

### Water mask correction

Water tiles (label 0) are reassigned to the union of all non-water geometry
so that the sea / large lakes do not leave white gaps in the map.

In [ ]:
not_water = gdf.loc[gdf["label"]!=0,"geometry"].union_all()
gdf.loc[gdf["label"]==0, "geometry"] = gdf.loc[gdf["label"]==0, "geometry"].apply(lambda g: g.difference(not_water))
gdf = gdf[gdf.geometry.notna()]

gdf.geometry = gdf.geometry.apply(lambda g: round_coords(g, decimals=4))
gdf.geometry = gdf.geometry.buffer(0)
gdf = gdf[gdf.geometry.is_valid]

## Creating the map

### Full-extent map (zoom level 5)

This is the primary output map showing all clustered regions across the covered area.
Each cluster is rendered as a semi-transparent polygon layer with a toggle in the layer control.

In [ ]:
m = folium.Map(location=[51.4, 10.7], zoom_start=5, tiles=None)

# Create a feature group for each cluster containing all the shapes
for label, gruppe in gdf.groupby("label"):
    coloured_name = generate_coloured_name(
        f"Cluster: {int(label+1)}", custom_colors_medium[int(label)]
    )
    folium.GeoJson(
        gruppe.__geo_interface__, name=coloured_name, style_function=style
    ).add_to(m)

# Create a feature group for the "representative positions"
extra_positions = folium.FeatureGroup(name="Orte aus den Renderings")
for cluster, positionslist in representative_positions.items():
    for posindex, position in enumerate(positionslist):
        extra_positions.add_child(
            folium.CircleMarker(
                (position[1], position[0]),
                popup=f"Cluster&nbsp;{cluster+1},<br>Beispiel&nbsp;{posindex+1}",
                fill_color=f"{custom_colors_dark[cluster]}",
                fill_opacity=0.5,
                color="white",
                weight=3,
                radius=15
            )
        )

# Create a feature group for two locations where photos were taken

photo_positions_dict = {"Braemar": (57.01, -3.397), "Schönau": (47.78623, 7.89445)}
photo_positions = folium.FeatureGroup(name="Photo Positions")
for name, location in photo_positions_dict.items():
    photo_positions.add_child(folium.Marker(location, icon=x_icon_big))


# Choose a tile layer with a beautiful shading and lesser text
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, USGS, NOAA",
    name="Esri Topo",
    control=False,
).add_to(m)

# Add the created features to the map
extra_positions.add_to(m)
# m.location = photo_positions_dict["Braemar"]
# m.zoom_start = 13
# photo_positions.add_to(m)
folium.map.LayerControl("topright", collapsed=False).add_to(m)
MousePosition().add_to(m)


m.save("output/maps_html/geo_documentation_europe_v3.html")

In [ ]:
# This is a safer way to get colored shapes
# We first convert the list above to a dict
# so we can make use of gray if the index is not found
def style(feature):
    colordict = {index: value for index, value in enumerate(custom_colors_medium)}
    farbe = colordict.get(feature["properties"]["label"], "gray")
    return {"fillColor": farbe, "color": farbe, "fillOpacity": 0.5}

In [ ]:
import copy

### Regional example map (zoom level 7)

A zoomed-in version of the same map, centered on a representative sub-region.
Useful for inspecting cluster boundaries at finer detail.

In [ ]:
m = folium.Map(location=[51.4, 10.7], zoom_start=7, tiles=None)

# Create a feature group for each cluster containing all the shapes
for label, gruppe in gdf.groupby("label"):
    coloured_name = generate_coloured_name(
        f"Cluster: {int(label+1)}", custom_colors_medium[int(label)]
    )
    folium.GeoJson(
        gruppe.__geo_interface__, name=coloured_name, style_function=style
    ).add_to(m)

# Create a feature group for the "representative positions"
extra_positions = folium.FeatureGroup(name="Representative Positions")
for cluster, positionslist in representative_positions.items():
    for posindex, position in enumerate(positionslist):
        extra_positions.add_child(
            folium.Marker(
                (position[1], position[0]),
                popup=f"Cluster&nbsp;{cluster+1},<br>Beispiel&nbsp;{posindex+1}",
                icon=x_icon[cluster],
            )
        )

# Create a feature group for two locations where photos were taken

photo_positions_dict = {"Braemar": (57.01, -3.397), "Schönau": (47.78623, 7.89445)}
photo_positions = folium.FeatureGroup(name="Photo Positions")
for name, location in photo_positions_dict.items():
    photo_positions.add_child(folium.Marker(location, icon=x_icon_big))


# Choose a tile layer with a beautiful shading and lesser text
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, USGS, NOAA",
    name="Esri Topo",
    control=False,
).add_to(m)

# Add the created features to the map
for locname in photo_positions_dict.keys():
    m_exp = copy.deepcopy(m)    

    m_exp.location = photo_positions_dict[locname]
    photo_positions.add_to(m_exp)

    filename_location = replace_umlaute(locname.lower())
    m_exp.save(f"output/maps_html/geo_documentation_{filename_location}.html")

